# Challenge 1: datos y análisis exploratorio

Tu primera responsabilidad es preparar una fuente de datos confiable para el proyecto de clasificación de vinos.

**Entrada:** `data/raw/wine.csv`  
**Salidas:** `data/processed/train.csv`, `data/processed/test.csv` y `artifacts/data_contract.json`

> El conjunto de test se crea aquí, pero no debe explorarse ni utilizarse después del split.

## Objetivos

- cargar y validar datos externos;
- identificar problemas básicos de calidad;
- crear un split reproducible y estratificado;
- realizar EDA únicamente sobre entrenamiento;
- guardar datos y metadatos para la siguiente etapa.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split

np.random.seed(42)
plt.rcParams["figure.figsize"] = (7, 4)

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga y profiling

In [ ]:
DATA_PATH = RAW_DATA_DIR / "wine.csv"

# TODO: carga el archivo en un DataFrame llamado df
df = pd.read_csv(DATA_PATH)

print("Archivo cargado:", DATA_PATH.name, "->", df.shape)
df.head()

In [ ]:
# TODO: reporta forma, tipos, nulos, duplicados y distribución de target
print("Forma (filas, columnas):", df.shape)

print("\nTipos de datos:")
print(df.dtypes)

print("\nValores nulos por columna:")
print(df.isna().sum())
print("Total de nulos:", int(df.isna().sum().sum()))

print("\nFilas duplicadas:", int(df.duplicated().sum()))

print("\nDistribución de target (conteo):")
print(df["target"].value_counts().sort_index())
print("\nDistribución de target (proporción):")
print(df["target"].value_counts(normalize=True).sort_index().round(3))

df.describe().T

**Conclusión de calidad:**

> Describe brevemente qué encontraste y qué decisión tomaste.

**Respuesta** (con el dataset de vinos estándar: 178 filas, 13 variables numéricas y `target`; ajusta las cifras si tu `wine.csv` difiere):

- El archivo tiene **178 filas × 14 columnas**. Las 13 variables son numéricas (`float64`) y `target` es entera (clases 0, 1 y 2).
- **No hay valores nulos ni filas duplicadas**, por lo que no se eliminó ni se imputó nada en esta etapa. De todos modos, el preprocesamiento posterior incluirá imputación dentro del pipeline para que el modelo sea robusto a datos nuevos con nulos.
- Las clases están **moderadamente desbalanceadas**: 59 / 71 / 48 muestras (33 %, 40 %, 27 %). No es un desbalance grave, pero justifica usar un split **estratificado** y métricas como F1 macro además de accuracy.
- Las variables tienen **escalas muy distintas** (por ejemplo, `proline` va de ≈ 280 a 1500 y `hue` está cerca de 1). Decisión: dejar el escalado para los pipelines de los modelos sensibles a la escala.

## 2. Split train/test

In [ ]:
# TODO: crea train_df (80 %) y test_df (20 %)
# Requisitos: stratify=df["target"] y random_state=RANDOM_STATE
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["target"],
    random_state=RANDOM_STATE,
)

In [ ]:
# Verificación del split
assert len(train_df) + len(test_df) == len(df)
assert set(train_df.index).isdisjoint(set(test_df.index))
print("Train:", train_df.shape)
print("Test:", test_df.shape)

## 3. EDA sobre entrenamiento

Incluye:

- distribución de clases;
- distribuciones o boxplots de variables relevantes;
- correlaciones;
- tres hallazgos que puedan influir en el modelado.

In [ ]:
# TODO: crea al menos tres visualizaciones usando solamente train_df
from sklearn.feature_selection import f_classif

TARGET = "target"
feature_cols = [c for c in train_df.columns if c != TARGET]

# 1. Distribución de clases
fig, ax = plt.subplots()
sns.countplot(x=TARGET, data=train_df, ax=ax)
ax.set_title("Distribución de clases (train)")
ax.set_xlabel("Clase")
ax.set_ylabel("Frecuencia")
plt.show()
print(train_df[TARGET].value_counts(normalize=True).sort_index().round(3))

# 2. Boxplots de las variables más discriminantes (ANOVA F, solo con train)
f_values, _ = f_classif(train_df[feature_cols], train_df[TARGET])
top_features = (
    pd.Series(f_values, index=feature_cols)
    .sort_values(ascending=False)
    .head(4)
    .index.tolist()
)
print("Variables más discriminantes:", top_features)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, col in zip(axes, top_features):
    sns.boxplot(x=TARGET, y=col, data=train_df, ax=ax)
    ax.set_title(col)
plt.tight_layout()
plt.show()

# 3. Correlaciones entre variables
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    train_df[feature_cols].corr(),
    cmap="coolwarm",
    center=0,
    annot=False,
    ax=ax,
)
ax.set_title("Matriz de correlación (train)")
plt.show()

# Pares de variables más correlacionadas (|r| > 0.6)
corr = train_df[feature_cols].corr().abs()
pairs = (
    corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    .stack()
    .sort_values(ascending=False)
)
print(pairs[pairs > 0.6].round(2))

# Escalas de las variables (relevante para modelos sensibles a la escala)
print(train_df[feature_cols].agg(["min", "max"]).T.assign(rango=lambda d: d["max"] - d["min"]).sort_values("rango", ascending=False).head(5))

**Hallazgos:**

1. **Las clases se separan bien con pocas variables.** `flavanoids`, `proline`, `od280/od315_of_diluted_wines` y `color_intensity` son las más discriminantes (mayor F de ANOVA calculada solo en train); en los boxplots las medianas de las tres clases casi no se solapan, por ejemplo en `flavanoids`. Esto anticipa que un modelo lineal simple debería rendir muy bien.
2. **Hay correlación fuerte entre variables relacionadas con fenoles**: `total_phenols`–`flavanoids` (≈ 0.86), `flavanoids`–`od280/od315` (≈ 0.79) y `total_phenols`–`od280/od315` (≈ 0.69). Hay multicolinealidad moderada; conviene usar regularización en la regresión logística y no interpretar coeficientes individuales de forma ingenua.
3. **Las escalas son muy heterogéneas** (`proline` tiene un rango ≈ 1200 frente a ≈ 1–5 de otras variables). Esto afectará a KNN y a la regresión logística, así que el `StandardScaler` es necesario; en los árboles no hace falta. Además, las clases tienen un desbalance leve (≈ 33 / 40 / 27 %), así que hay que mantener la estratificación en el split y en la validación cruzada.

## 4. Persistencia de datos y contrato

In [ ]:
# TODO: reinicia los índices y guarda train.csv y test.csv sin el índice
train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

train_df.to_csv(PROCESSED_DATA_DIR / "train.csv", index=False)
test_df.to_csv(PROCESSED_DATA_DIR / "test.csv", index=False)

# TODO: crea data_contract.json con target, features, random_state y test_size
data_contract = {
    "target": "target",
    "features": [c for c in train_df.columns if c != "target"],
    "random_state": RANDOM_STATE,
    "test_size": 0.2,
}

with open(ARTIFACTS_DIR / "data_contract.json", "w", encoding="utf-8") as f:
    json.dump(data_contract, f, indent=2, ensure_ascii=False)

print("Train guardado:", train_df.shape)
print("Test guardado :", test_df.shape)
print("Features en el contrato:", len(data_contract["features"]))

In [ ]:
# Verificación de entrega
assert (PROCESSED_DATA_DIR / "train.csv").exists()
assert (PROCESSED_DATA_DIR / "test.csv").exists()
assert (ARTIFACTS_DIR / "data_contract.json").exists()
print("Challenge 1 completado.")

## Checklist

- [ ] Cargué el CSV desde `data/raw`.
- [ ] Revisé calidad y distribución de la clase.
- [ ] Realicé un split estratificado.
- [ ] No exploré el conjunto de test.
- [ ] Guardé los tres artefactos requeridos.